# 01 — UK-DALE 60-day EDA

Explores the synthesized or downloaded UK-DALE House 1 subset used for training (first 46 days) and the simulation window (last 14 days).

Run `scripts/fetch_ukdale_subset.py` first to populate `data/ukdale/`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from aerogrid.config import UKDALE_DIR, UKDALE_TRAIN_START, UKDALE_TEST_START, UKDALE_TEST_END

mains = pd.read_parquet(UKDALE_DIR / "house_1" / "mains_1hz.parquet")
dish = pd.read_parquet(UKDALE_DIR / "house_1" / "dishwasher_6s.parquet")
wash = pd.read_parquet(UKDALE_DIR / "house_1" / "washing_machine_6s.parquet")
onsets = pd.read_parquet(UKDALE_DIR / "house_1" / "onsets.parquet")
print(f"mains: {len(mains):,} rows | dishwasher: {len(dish):,} | washer: {len(wash):,}")
print(f"onsets: {onsets['appliance'].value_counts().to_dict()}")

In [ ]:
# Daily mean power (aggregate + per-appliance) with train/test split marker
fig, ax = plt.subplots(figsize=(12,4))
for df, label in [(mains, "mains"), (dish, "dishwasher"), (wash, "washing_machine")]:
    daily = df.set_index("timestamp").resample("1D")["power_w"].mean()
    ax.plot(daily.index, daily.values, label=label, linewidth=1)
ax.axvline(UKDALE_TEST_START, color="red", linestyle="--", label="train/test split")
ax.set_ylabel("power (W, daily mean)"); ax.legend(); ax.set_title("UK-DALE House 1: daily mean power")
plt.tight_layout(); plt.show()

In [ ]:
# Onset counts per appliance per hour-of-day, split by train/test
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, split in zip(axes, ["train", "test"]):
    sub = onsets[onsets["split"] == split]
    for app in ("dishwasher", "washing_machine"):
        hod = sub[sub["appliance"] == app]["timestamp"].dt.hour
        ax.hist(hod, bins=range(25), alpha=0.6, label=app)
    ax.set_title(f"{split}  (n={len(sub)})"); ax.set_xlabel("hour of day"); ax.legend()
axes[0].set_ylabel("# onsets"); plt.tight_layout(); plt.show()

In [ ]:
# Per-appliance cycle power distribution — histogram of non-zero 6 s samples
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df, name in zip(axes, [dish, wash], ["dishwasher", "washing_machine"]):
    vals = df.loc[df["power_w"] > 50, "power_w"]
    ax.hist(vals, bins=60); ax.set_title(f"{name}: {len(vals):,} 'on' samples")
    ax.set_xlabel("power (W)"); ax.set_ylabel("count"); ax.set_yscale("log")
plt.tight_layout(); plt.show()